# Lossfunk Autoresearch — LeWM / Hi-LeWM End-to-End Experiment

This notebook is the **execution layer** for the research work already developed in Codex.

It is designed to:
1. recreate the CUDA environment,
2. verify the official LeWM checkpoint and PushT dataset,
3. run the existing Codex instrumentation,
4. reproduce the Flat LeWM baseline,
5. run Flat Receding-Horizon vs Hierarchical experiments,
6. perform horizon/compute analysis,
7. preserve raw traces and reproducibility artifacts.

**Important:** this notebook does not invent numerical horizons, seeds, compute budgets, or hierarchy settings. Those must come from the existing Codex protocol before scientific runs.


In [ ]:
# 0 — Runtime / GPU check
import os, sys, subprocess, platform, shutil, json, hashlib, time
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA runtime:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
except Exception as e:
    print("Torch check failed:", repr(e))

print(subprocess.run(["df","-h","/content"], capture_output=True, text=True).stdout)


## 1 — Frozen research definitions

- **H:** number of primitive environment action steps in the overall task horizon.
- **C:** `(N_WM, N_P)`.
- **Planner evaluation:** one candidate planning object actually scored/evaluated.
- **Feedback parity:** external observation and external replanning frequency are matched.
- **Model parity:** exactly the same trained world model/checkpoint is used.
- **Training parity:** same underlying data/distribution; no privileged expert/ground-truth subgoal annotations in the primary experiment.
- **Primary comparison:** Flat Receding Horizon vs Hierarchical.
- **Delta:** `S_hier(H,C) - S_flat_RH(H,C)`.

The notebook must not silently change these definitions.


In [ ]:
# 2 — Experimental protocol
# COPY ONLY values already approved/frozen in the Codex research workspace.
PROTOCOL = {
    "env": "PushT",
    "baseline_policy": "pusht/lewm",
    "hierarchical_policy": "pusht/hi_lewm",

    "horizons": None,
    "seeds": None,
    "episodes_per_condition": None,

    "obs_frequency": None,
    "replan_frequency": None,

    "target_N_WM": None,
    "target_N_P": None,

    "baseline": {
        "horizon": 5,
        "action_block": 5,
        "receding_horizon": 5,
        "cem_samples": 300,
        "cem_iterations": 30,
        "elite_count": 30,
        "batch_size": 1,
        "effective_internal_horizon_steps": 25,
        "effective_replan_interval_steps": 25,
    }
}

print(json.dumps(PROTOCOL, indent=2))
missing = [x for x in ("horizons","seeds","episodes_per_condition") if PROTOCOL[x] is None]
if missing:
    print("SCIENTIFIC RUNS BLOCKED until these are populated from the Codex protocol:", missing)


## 3 — Create the research workspace

The Colab runtime is disposable, so the notebook writes all experiment outputs under `/content/lossfunk_autoresearch/research/results`.

For a long experiment, copy the resulting `lossfunk_autoresearch` folder to Google Drive after each major stage.


In [ ]:
ROOT = Path("/content/lossfunk_autoresearch")
ROOT.mkdir(exist_ok=True)
RESULTS = ROOT/"research"/"results"
TRACE = RESULTS/"traces"
ARTIFACTS = RESULTS/"artifacts"
for p in (RESULTS, TRACE, ARTIFACTS):
    p.mkdir(parents=True, exist_ok=True)
print(ROOT)


## 4 — Bring in the existing Codex workspace

Recommended: upload the **Codex workspace ZIP** to Colab and extract it here. This preserves the existing instrumentation, logs, protocol, and research artifacts instead of rebuilding them.

If you instead clone fresh repositories, you must also transfer the Codex instrumentation and research files before running experiments.


In [ ]:
# Upload the Codex workspace ZIP if you have it.
# This cell is intentionally interactive; after upload, extract it into ROOT.
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded))


In [ ]:
# Example extraction cell — adjust ZIP name if needed.
import zipfile, glob, os
zips = glob.glob("/content/*.zip")
print("ZIP files:", zips)

if zips:
    zip_path = zips[0]
    extract_to = Path("/content/codex_upload")
    extract_to.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_to)
    print("Extracted:", extract_to)
else:
    print("No ZIP found. If repositories are being cloned instead, continue to the next cell.")


In [ ]:
# Inspect uploaded workspace and locate the existing instrumentation.
for p in sorted(Path("/content").glob("**/BASELINE_PROTOCOL.md"))[:10]:
    print("BASELINE_PROTOCOL:", p)
for p in sorted(Path("/content").glob("**/HUMAN_INTERVENTIONS.md"))[:10]:
    print("HUMAN_INTERVENTIONS:", p)
for p in sorted(Path("/content").glob("**/instrumentation*"))[:20]:
    print("Instrumentation:", p)


## 5 — Clone exact upstream repositories if needed

These are the repositories used by the Codex investigation:

- `lucas-maes/le-wm`
- `NiccoloCase/Hi-LeWM`

The exact commit used by the research must be recorded before the scientific run.


In [ ]:
REPO_ROOT = ROOT/"repos"
REPO_ROOT.mkdir(exist_ok=True)

repos = {
    "le-wm": "https://github.com/lucas-maes/le-wm.git",
    "Hi-LeWM": "https://github.com/NiccoloCase/Hi-LeWM.git",
}
for name, url in repos.items():
    p = REPO_ROOT/name
    if not p.exists():
        subprocess.run(["git","clone",url,str(p)], check=True)
    sha = subprocess.check_output(["git","rev-parse","HEAD"], cwd=p, text=True).strip()
    print(name, sha)


## 6 — Install the compatibility environment

The local Codex investigation found that the LeWM evaluator expects `stable_worldmodel.wm.utils`, which is available in `stable-worldmodel==0.1.0`, while the tested `0.0.6` environment lacked that API.

The validated candidate pair was:
- `stable-worldmodel==0.1.0`
- `stable-pretraining==0.1.6`

We intentionally avoid blindly installing training extras that could upgrade `stable-pretraining` beyond the tested compatibility pair.


In [ ]:
%pip install -q "stable-worldmodel==0.1.0" "stable-pretraining==0.1.6"
%pip install -q hydra-core omegaconf scikit-learn torchvision h5py zstandard huggingface_hub


In [ ]:
import torch, stable_worldmodel as swm
print("torch:", torch.__version__)
print("stable-worldmodel:", getattr(swm, "__version__", "unknown"))
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA GPU required for the official evaluator."

from stable_worldmodel.wm import utils as swm_utils
print("load_pretrained API:", swm_utils.load_pretrained)


## 7 — Install source trees

Install the checked-out repositories without allowing them to silently replace the pinned evaluation dependencies.


In [ ]:
%cd /content/lossfunk_autoresearch/repos/le-wm
%pip install -q -e . --no-deps
%cd /content/lossfunk_autoresearch/repos/Hi-LeWM
%pip install -q -e . --no-deps
%cd /content/lossfunk_autoresearch


In [ ]:
STABLEWM_HOME = Path("/content/stablewm")
os.environ["STABLEWM_HOME"] = str(STABLEWM_HOME)
os.environ["MUJOCO_GL"] = "egl"
STABLEWM_HOME.mkdir(exist_ok=True)
print("STABLEWM_HOME =", STABLEWM_HOME)


## 8 — Download the official LeWM PushT checkpoint

Official source:
`quentinll/lewm-pusht`

Files:
- `config.json`
- `weights.pt`

Do not use a third-party mirror for the primary experiment.


In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = "quentinll/lewm-pusht"
MODEL_DIR = STABLEWM_HOME/"pusht_hf"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

config_path = hf_hub_download(MODEL_REPO, "config.json", local_dir=str(MODEL_DIR))
weights_path = hf_hub_download(MODEL_REPO, "weights.pt", local_dir=str(MODEL_DIR))
print(config_path)
print(weights_path)


## 9 — Convert the official checkpoint

Use the **official conversion mechanism in the checked-out LeWM repository**. Do not guess a replacement loader.

First inspect the conversion utility and then run its documented command. The expected output from the established local setup is:

`$STABLEWM_HOME/pusht/lewm_object.ckpt`


In [ ]:
LEWM = REPO_ROOT/"le-wm"
print("Conversion-related files:")
for p in sorted(LEWM.rglob("*")):
    if p.is_file() and any(x in p.name.lower() for x in ("convert","checkpoint","weight")):
        print(p.relative_to(LEWM))


In [ ]:
# After running the official conversion command, verify the object checkpoint.
CKPT = STABLEWM_HOME/"pusht"/"lewm_object.ckpt"
if CKPT.exists():
    h = hashlib.sha256()
    with open(CKPT,"rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024), b""):
            h.update(chunk)
    print("Checkpoint:", CKPT)
    print("SHA256:", h.hexdigest())
else:
    print("Checkpoint not yet present:", CKPT)


## 10 — Download and verify the official PushT dataset

Official archive:
`pusht_expert_train.h5.zst`

Known official SHA-256:
`7cfbd6d90fa2f27876379a5ff169715a36ed82edbda64f9e5b5bfa34d212f318`

The archive is about 13.1 GB and the extracted HDF5 about 46.3 GB.


In [ ]:
DATA_ARCHIVE = STABLEWM_HOME/"pusht_expert_train.h5.zst"
DATA_H5 = STABLEWM_HOME/"pusht_expert_train.h5"

hf_hub_download(
    "quentinll/lewm-pusht",
    "pusht_expert_train.h5.zst",
    local_dir=str(STABLEWM_HOME)
)

EXPECTED_SHA256 = "7cfbd6d90fa2f27876379a5ff169715a36ed82edbda64f9e5b5bfa34d212f318"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

actual = sha256_file(DATA_ARCHIVE)
print("SHA256:", actual)
assert actual == EXPECTED_SHA256, "Official dataset checksum mismatch."

if not DATA_H5.exists():
    subprocess.run(["zstd","-d","-f",str(DATA_ARCHIVE),"-o",str(DATA_H5)], check=True)

print("Dataset bytes:", DATA_H5.stat().st_size)


In [ ]:
import h5py
with h5py.File(DATA_H5,"r") as f:
    print("Dataset keys:", list(f.keys()))


# 11 — Instrumentation gate

The Codex instrumentation must capture:

- decision-time information `I_t`;
- observation and replanning events;
- action-buffer activity;
- CEM candidate counts/iterations;
- `N_P`;
- world-model calls;
- planning/model/episode timings;
- numeric actions at environment handoff;
- environment transitions;
- exact run metadata.

Do not change the planner or evaluator semantics just to add these measurements.


In [ ]:
# Locate instrumentation from the uploaded Codex workspace.
search_roots = [Path("/content/codex_upload"), ROOT]
for sr in search_roots:
    if sr.exists():
        for p in sorted(sr.rglob("*")):
            if p.is_file() and ("instrument" in p.name.lower() or "trace" in p.name.lower()):
                print(p)


# 12 — Baseline preflight

Before scientific data collection, verify:

- CUDA works;
- exact commits are recorded;
- package versions are recorded;
- checkpoint loads strictly;
- dataset checksum passes;
- intended PushT baseline config is used;
- instrumentation is active;
- no hierarchy code is used;
- the one-episode smoke test completes;
- traces contain all required events.

The smoke test is **not** a scientific result.


## 13 — Existing baseline configuration

The inspected baseline configuration is:

- CEM samples: 300
- CEM iterations: 30
- elites: 30
- batch size: 1
- `horizon=5`
- `action_block=5`
- `receding_horizon=5`
- effective internal horizon: 25 primitive action steps
- effective external replanning interval: 25 primitive action steps

These are the inspected baseline defaults; do not treat them as the final horizon-scaling protocol unless the Codex protocol says so.


# 14 — Instrumented one-episode smoke test

Run the **existing Codex instrumentation launcher** here.

Do not replace it with the plain upstream evaluator.

After it completes, verify:
- 50-step smoke configuration behaves as expected;
- observations are recorded;
- replans are recorded;
- CEM calls/candidates are recorded;
- WM calls are recorded;
- numeric actions and environment transitions are recorded.


In [ ]:
# Locate likely launchers. Use the exact launcher already developed by Codex.
for base in [Path("/content/codex_upload"), ROOT]:
    if base.exists():
        for p in sorted(base.rglob("*.py")):
            txt = ""
            try: txt = p.read_text(errors="ignore")
            except: pass
            if any(k in txt.lower() for k in ("instrumentation", "planning_calls", "wm_trace", "environment_transition")):
                print(p)


# 15 — Scientific baseline reproduction

Only after the smoke test passes:

Required raw artifacts:
- `baseline_config.yaml`
- `episodes.csv`
- `planning_calls.csv`
- `wm_trace.json`
- `baseline_information.json`

Each run also records:
- git commit;
- exact config;
- checkpoint/dataset identity;
- Python/PyTorch/CUDA;
- GPU;
- seed;
- exact command;
- environment version.

No scientific result should be inferred from the failed local CPU attempt.


In [ ]:
# Create a unique run directory.
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime("baseline_%Y%m%dT%H%M%SZ")
RUN_DIR = RESULTS/RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print(RUN_DIR)


## 16 — Baseline command

Use the exact instrumented baseline command from the Codex workspace here.

Do not guess CLI arguments. If the launcher differs after migration, inspect it first and preserve its semantics.


In [ ]:
# PLACEHOLDER: insert the existing Codex instrumented baseline command.
# Example shape only:
#
# !python <existing_launcher>.py --config-name=pusht policy=pusht/lewm ...
#
print("Waiting for the exact migrated Codex launcher.")


# 17 — Baseline information audit

From the actual trace determine the information consumed at each decision point:

- current pixels;
- goal pixels;
- history;
- previous actions;
- state/proprio fields;
- world-model input;
- retained planning context;
- any privileged information.

Do not count a field merely because it exists in an environment object; count what the baseline actually consumes.


# 18 — Freeze the primary comparison

After baseline inspection:

1. determine feasible task horizons;
2. choose short/medium/long with a method-independent rule;
3. freeze H;
4. freeze external observation frequency;
5. freeze external replanning frequency;
6. freeze seeds/episode count;
7. freeze success criterion;
8. freeze information parity;
9. freeze shared checkpoint;
10. freeze compute accounting.

Do not use primary results to retroactively select these values.


# 19 — Primary experiment: Flat RH vs Hierarchical

For every `(H, seed)`:

**Flat RH**
- shared LeWM world model;
- same externally available information;
- same external observation/replanning frequency;
- candidate-level `N_P`.

**Hierarchical**
- same shared LeWM world model;
- internal high-level candidate subgoals;
- internal low-level candidate plans;
- `N_P^Hier = N_P^HL + N_P^LL`;
- log HL and LL separately.

Primary compute accounting:
`C = (N_WM, N_P)`.

Do not claim equal total FLOPs or equal planner parameter count unless separately demonstrated.


In [ ]:
# 20 — Build the approved condition matrix
import pandas as pd

assert PROTOCOL["horizons"] is not None
assert PROTOCOL["seeds"] is not None
assert PROTOCOL["episodes_per_condition"] is not None

rows = []
for H in PROTOCOL["horizons"]:
    for seed in PROTOCOL["seeds"]:
        for method in ("flat_rh","hierarchical"):
            rows.append({
                "method": method,
                "H": H,
                "seed": seed,
                "episodes": PROTOCOL["episodes_per_condition"]
            })

conditions = pd.DataFrame(rows)
conditions


# 21 — Run primary campaign

Use the existing instrumented Flat and Hierarchical launchers.

For each condition:
- create a unique run directory;
- save exact stdout/stderr;
- save raw traces;
- save exact config;
- preserve failures;
- do not retune based on interim performance.

Technical failures and scientific failures must remain distinguishable.


In [ ]:
# 22 — Campaign runner placeholder
# Connect this to the existing Codex launchers only after their exact CLI/API is verified.
print("Campaign runner intentionally not guessed.")


# 23 — Aggregation

For every episode retain:

- method
- H
- seed
- success/failure
- final task-state error
- environment steps
- planning calls
- `N_WM`
- `N_P`
- `N_P^HL`
- `N_P^LL`
- planning/model/total inference time
- failure category

Primary quantity:
`Delta(H,C) = S_hier(H,C) - S_flat_RH(H,C)`.

Use independent episodes/seeds as the experimental units; do not treat individual environment steps as independent replicates.


In [ ]:
# 24 — Find result tables after the campaign
for p in sorted(RESULTS.rglob("episodes.csv")):
    print(p)


# 25 — Horizon scaling

Using only the pre-frozen horizon set, report:

- success vs H for Flat RH;
- success vs H for Hierarchical;
- Delta(H,C) vs H;
- compute quantities vs H;
- failure categories vs H.

Do not add/remove horizons after seeing primary results.


# 26 — Compute audit

For every condition report:

- `N_WM`;
- `N_P`;
- hierarchical HL/LL counts;
- planning iterations;
- FLOPs if available;
- wall-clock inference time;
- planner parameter counts;
- environment interactions.

If exact `N_WM` matching cannot be established, state that limitation instead of calling the comparison exactly compute-matched.


# 27 — Planned diagnostics

Only after the primary comparison is complete:

- oracle/noisy subgoals;
- constrained/unconstrained high-level search;
- reachability diagnostics;
- additional environment, if part of the approved plan.

Oracle subgoals are privileged and must remain diagnostic, not primary evidence.


# 28 — Final raw research package

Preserve:

### Research
question, hypothesis, definitions, frozen protocol, fairness controls.

### Execution
commits, package versions, checkpoint/dataset hashes, commands, raw traces, failed runs.

### Analysis
episode tables, uncertainty, compute audit, failure categories, plots.

### AI/human record
what Codex implemented/executed, what ChatGPT contributed, human decisions/interventions, limitations.

Do not overwrite raw AI-generated artifacts to make them presentation-ready. Make separate presentation material.


In [ ]:
# 29 — Final inventory
for p in sorted(ROOT.rglob("*")):
    if p.is_file():
        print(p.relative_to(ROOT))


# 30 — Completion checklist

- [ ] baseline reproduced
- [ ] baseline information audit completed
- [ ] H frozen before primary results
- [ ] observation/replanning matched
- [ ] same WM/checkpoint
- [ ] training/data parity documented
- [ ] N_P counted with candidate-level definition
- [ ] N_WM explicitly defined from trace
- [ ] HL/LL counts separately logged
- [ ] independent seeds/episodes
- [ ] technical vs scientific failures separated
- [ ] raw traces preserved
- [ ] compute audit reported
- [ ] horizon scaling completed
- [ ] diagnostics kept separate
- [ ] limitations/uncertainty reported
